In [19]:

#sk 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold

import pickle

import pandas as pd

from pathlib import Path

from tqdm import tqdm

In [20]:
with open("/home/wuhlmann/BA/data/processed_data/cell_states/q_pred_landcover_0903_135428_cell_states.p", "rb") as f: 
    cell_state_dict = pickle.load(f)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1527e87df130>>
Traceback (most recent call last):
  File "/storage/vast-gfz-hpc-01/home/wuhlmann/miniforge3/envs/lamah-ce_lstm/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [28]:

ts_path = Path("/home/wuhlmann/BA/data/raw_data/2_LamaH-CE_daily/B_basins_intermediate_all/2_timeseries/daily")

# normalize cell states, since they are not necessarily on the same scale
cell_states = StandardScaler().fit_transform(cell_state_dict[str(1)]["c_last"])

ts = pd.read_table(ts_path / f"ID_{1}.csv", header=0, sep=";")

ts["date"] = pd.to_datetime(
			ts[["YYYY", "MM", "DD"]].rename(columns={"YYYY": "year", "MM": "month", "DD": "day"})
		)
ts.set_index("date", inplace=True)

# drop all columns except for 2m_mean_temp
temp = ts[["swe"]] / 1000

# Cut temperature ts to match start and end date of the discharge ts, then remove the warmup period by only taking the len(c) days starting from the end. 
temp = temp.iloc[-(len(cell_states)):]

# set l1 to 1, to confirm theory that all dimensions being used to represent swe, is because the model needs to compute a very high sum of swe 
model = ElasticNet(random_state=1277, l1_ratio=0.15, max_iter=5000)
kfold = KFold(n_splits=5)
folds = list(kfold.split(cell_states, temp.values))

best_r2 = -100
best_model_id = None
model_list = []

# Do 5 fold validation and let the best model predict (probe) on the entire timeseries.
for i, f in tqdm(zip([0, 1, 2, 3, 4], folds)):
	
	train_ids = f[0]
	test_ids = f[1]

	target = temp.iloc[test_ids]

	model.fit(X=cell_states[train_ids], y=temp.iloc[train_ids])
	pred = model.predict(X=cell_states[test_ids])

	# Collect all models.
	model_list.append(model)

	r2 = r2_score(target, pred)

	if r2 > best_r2:
		best_r2 = r2
		best_model_id = i

best_model = model_list[best_model_id]

total_pred = best_model.predict(cell_states)
total_r2 = r2_score(temp, total_pred)
total_mae = mean_absolute_error(temp, total_pred)

0it [00:00, ?it/s]

5it [00:00, 26.85it/s]


In [29]:
best_r2

0.3035706877708435